# Standard Variational Autoencoder (VAE)

In this notebook, we implement a standard Variational Autoencoder (VAE) on simple data (Gaussian mixture or MNIST). This serves as a foundation for understanding **Sequential VAEs** like LFADS, which we'll study next.

## What is a VAE?

A VAE is a generative model that learns to:
1. **Encode** high-dimensional data into a low-dimensional latent space
2. **Decode** samples from the latent space back to the data space
3. Balance reconstruction accuracy with a regularization penalty (KL divergence)

**Key insight**: Instead of learning a deterministic mapping, VAEs learn **distributions** over latent variables, enabling both inference and generation.

**References**:
- Kingma & Welling (2013). "Auto-Encoding Variational Bayes" https://arxiv.org/abs/1312.6114
- PyTorch Official VAE Example: https://github.com/pytorch/examples/tree/main/vae

## Part 0: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Normal

import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange, tqdm

# Set random seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 1: Generate or Load Data

We'll use a simple 2D Gaussian mixture model for visualization and interpretability.

In [ ]:
# Generate 2D Gaussian mixture data
def generate_gaussian_mixture(n_samples=5000, n_components=3):
    """Generate data from a Gaussian mixture model."""
    np.random.seed(0)
    
    # Define component means
    means = np.array([
        [-2, -2],
        [2, 2],
        [2, -2]
    ])[:n_components]
    
    # Generate data
    samples = []
    for i in range(n_components):
        n = n_samples // n_components
        component_samples = np.random.randn(n, 2) * 0.5 + means[i]
        samples.append(component_samples)
    
    data = np.vstack(samples)
    np.random.shuffle(data)
    return torch.FloatTensor(data)

# Generate data
data = generate_gaussian_mixture(n_samples=5000, n_components=3)
print(f"Data shape: {data.shape}")

# Create DataLoader
dataset = TensorDataset(data)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(dataset, batch_size=128, shuffle=False)

# Plot original data
plt.figure(figsize=(6, 6))
plt.scatter(data[:, 0], data[:, 1], alpha=0.5, s=10)
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Original Data Distribution')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()

## Part 2: VAE Model Architecture

### Part 2a: Encoder

The encoder maps data $x$ to a distribution over latent variables:
$$q_\phi(z|x) = \mathcal{N}(\mu_\phi(x), \sigma_\phi^2(x))$$

In [ ]:
class Encoder(nn.Module):
    """Encoder: maps data to latent variable distribution.
    
    Maps input x to mean and log-variance of q(z|x).
    """
    def __init__(self, input_dim=2, hidden_dim=32, latent_dim=2):
        super(Encoder, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Output: mean and log-variance of latent distribution
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
    
    def forward(self, x):
        """Forward pass.
        
        Args:
            x: input data (batch_size, input_dim)
        
        Returns:
            mu: mean of q(z|x) (batch_size, latent_dim)
            logvar: log-variance of q(z|x) (batch_size, latent_dim)
        """
        h = F.relu(self.fc1(x))
        h = F.relu(self.fc2(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

### Part 2b: Reparameterization Trick

To backpropagate through sampling, we use the **reparameterization trick**:
$$z = \mu + \epsilon \odot \sigma, \quad \epsilon \sim \mathcal{N}(0, I)$$

This allows gradients to flow through the stochastic sampling process.

In [ ]:
def reparameterize(mu, logvar):
    """Reparameterization trick: sample z ~ q(z|x).
    
    Args:
        mu: mean of q(z|x)
        logvar: log-variance of q(z|x)
    
    Returns:
        z: sample from q(z|x)
    """
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z = mu + eps * std
    return z

### Part 2c: Decoder

The decoder maps latent variables back to data space:
$$p_\theta(x|z)$$

In [ ]:
class Decoder(nn.Module):
    """Decoder: maps latent variables to data.
    
    Maps latent z to reconstructed data x_hat.
    """
    def __init__(self, latent_dim=2, hidden_dim=32, output_dim=2):
        super(Decoder, self).__init__()
        
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, z):
        """Forward pass.
        
        Args:
            z: latent variables (batch_size, latent_dim)
        
        Returns:
            x_recon: reconstructed data (batch_size, output_dim)
        """
        h = F.relu(self.fc1(z))
        h = F.relu(self.fc2(h))
        x_recon = self.fc_out(h)
        return x_recon

### Part 2d: Complete VAE

Combine encoder, reparameterization, and decoder into a single VAE module.

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder."""
    
    def __init__(self, input_dim=2, hidden_dim=32, latent_dim=2):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)
        self.latent_dim = latent_dim
    
    def forward(self, x):
        """Forward pass through VAE.
        
        Args:
            x: input data (batch_size, input_dim)
        
        Returns:
            x_recon: reconstructed data
            mu: mean of q(z|x)
            logvar: log-variance of q(z|x)
            z: sampled latent variables
        """
        # Encode
        mu, logvar = self.encoder(x)
        
        # Reparameterize
        z = reparameterize(mu, logvar)
        
        # Decode
        x_recon = self.decoder(z)
        
        return x_recon, mu, logvar, z

## Part 3: ELBO Loss Function

The VAE is trained by maximizing the **Evidence Lower BOund (ELBO)**:

$$\mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_\text{Reconstruction} - \underbrace{D_{KL}(q_\phi(z|x) || p(z))}_\text{KL Regularization}$$

Where:
- **Reconstruction term**: How well can we reconstruct x from z?
- **KL term**: How close is the learned posterior to the prior p(z) = N(0, I)?

In [ ]:
def vae_loss(x_recon, x, mu, logvar):
    """Compute VAE loss (negative ELBO).
    
    Args:
        x_recon: reconstructed data
        x: original data
        mu: mean of q(z|x)
        logvar: log-variance of q(z|x)
    
    Returns:
        loss: negative ELBO (scalar)
        recon_loss: reconstruction loss component
        kl_loss: KL divergence component
    """
    # Reconstruction loss: mean squared error
    recon_loss = F.mse_loss(x_recon, x, reduction='mean')
    
    # KL divergence: D_KL(q(z|x) || p(z)) where p(z) = N(0, I)
    # KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    
    # Total loss
    loss = recon_loss + kl_loss
    
    return loss, recon_loss, kl_loss

## Part 4: Training Loop

In [ ]:
# Initialize model
model = VAE(input_dim=2, hidden_dim=32, latent_dim=2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training parameters
num_epochs = 50
train_losses = []
recon_losses = []
kl_losses = []

# Training loop
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    epoch_recon = 0
    epoch_kl = 0
    
    for batch_idx, (x,) in enumerate(train_loader):
        x = x.to(device)
        
        # Forward pass
        x_recon, mu, logvar, z = model(x)
        
        # Compute loss
        loss, recon, kl = vae_loss(x_recon, x, mu, logvar)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_recon += recon.item()
        epoch_kl += kl.item()
    
    # Average loss over epoch
    avg_loss = epoch_loss / len(train_loader)
    avg_recon = epoch_recon / len(train_loader)
    avg_kl = epoch_kl / len(train_loader)
    
    train_losses.append(avg_loss)
    recon_losses.append(avg_recon)
    kl_losses.append(avg_kl)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}, Recon: {avg_recon:.4f}, KL: {avg_kl:.4f}')

print('\nTraining complete!')

## Part 5: Analysis and Results

### Part 5a: Training curves

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_losses, label='Total Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(recon_losses, label='Reconstruction Loss', color='orange')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Reconstruction Loss (MSE)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(kl_losses, label='KL Divergence', color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('KL Divergence')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Part 5b: Reconstruct data

In [ ]:
# Get reconstructions
model.eval()
with torch.no_grad():
    x_sample = data[:1000].to(device)
    x_recon, _, _, _ = model(x_sample)
    x_recon = x_recon.cpu()

# Plot original vs reconstructed
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(data[:1000, 0], data[:1000, 1], alpha=0.5, s=10)
axes[0].set_xlabel('Dimension 1')
axes[0].set_ylabel('Dimension 2')
axes[0].set_title('Original Data')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

axes[1].scatter(x_recon[:, 0], x_recon[:, 1], alpha=0.5, s=10, color='orange')
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')
axes[1].set_title('Reconstructed Data')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

### Part 5c: Visualize latent space

In [ ]:
# Encode data to latent space
with torch.no_grad():
    x_sample = data[:1000].to(device)
    mu, _ = model.encoder(x_sample)
    mu = mu.cpu()

# Plot latent space
plt.figure(figsize=(8, 8))
plt.scatter(mu[:, 0], mu[:, 1], alpha=0.5, s=20, c=data[:1000, 0], cmap='viridis')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Learned Latent Space (colored by original x-coordinate)')
plt.colorbar(label='Original X')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()

### Part 5d: Generate new samples

Sample from the prior p(z) = N(0, I) and decode to generate new data points.

In [ ]:
# Generate new samples from prior
with torch.no_grad():
    z_samples = torch.randn(500, 2).to(device)  # Sample from N(0, I)
    x_generated = model.decoder(z_samples)
    x_generated = x_generated.cpu()

# Plot generated samples
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(data[:500, 0], data[:500, 1], alpha=0.5, s=10, label='Original')
axes[0].set_xlabel('Dimension 1')
axes[0].set_ylabel('Dimension 2')
axes[0].set_title('Original Data')
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

axes[1].scatter(x_generated[:, 0], x_generated[:, 1], alpha=0.5, s=10, color='green', label='Generated')
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')
axes[1].set_title('Generated from Prior p(z) = N(0, I)')
axes[1].grid(True, alpha=0.3)
axes[1].axis('equal')

plt.tight_layout()
plt.show()

print(f"Original data range: x=[{data[:500, 0].min():.2f}, {data[:500, 0].max():.2f}], y=[{data[:500, 1].min():.2f}, {data[:500, 1].max():.2f}]")
print(f"Generated data range: x=[{x_generated[:, 0].min():.2f}, {x_generated[:, 0].max():.2f}], y=[{x_generated[:, 1].min():.2f}, {x_generated[:, 1].max():.2f}]")

## Summary

In this notebook, we implemented a **Standard VAE** with the following key components:

1. **Encoder**: Maps data x → latent distribution q(z|x)
2. **Reparameterization trick**: Enables backpropagation through sampling
3. **Decoder**: Maps latent z → reconstructed data
4. **ELBO loss**: Balances reconstruction and KL regularization
5. **Generation**: Can sample from prior p(z) to generate new data

## Next Steps

In the next notebook (Part 3, 08_lfads.ipynb), we'll extend this to **Sequential VAE (LFADS)** by:
- Adding **recurrent encoders/decoders** for time-series data
- Modeling **dynamics** in the latent space
- Inferring **external inputs** driving the dynamics
- Applying to **neural spike train data**